# OAuth Endpoint Test Script

This script demonstrates OAuth2 client-credentials authentication and tests a URL endpoint using the obtained bearer token.

In [0]:
import requests
import json
from datetime import datetime

# ──────────────────────────────────────────────
# GitHub OAuth Test Configuration
# ──────────────────────────────────────────────
# Step 1: Create a Personal Access Token at:
#   https://github.com/settings/tokens (classic) or
#   https://github.com/settings/personal-access-tokens (fine-grained)
# Step 2: Store it in Databricks secrets:
#   dbutils.secrets.put(scope="my_scope", key="github_token", string_value="ghp_xxxxx")
# Step 3: Reference it below

# ── Option A: Quick test — paste your PAT directly (temporary, not for production) ──
GITHUB_TOKEN = ""  # ← DO NOT paste tokens here. Use Databricks secrets instead.

# Recommended: store your PAT in Databricks secrets, then read it:
#   dbutils.secrets.put(scope="<your_scope>", key="github_pat", string_value="ghp_xxx")
# Then uncomment:
# try:
#     GITHUB_TOKEN = dbutils.secrets.get(scope="<your_scope>", key="github_pat")
# except Exception:
#     pass

# ── Auto-switch endpoints based on token presence ──
if GITHUB_TOKEN:
    # Authenticated endpoints (5,000 req/hr, works with private repos)
    GITHUB_USER_URL  = "https://api.github.com/user"
    GITHUB_REPOS_URL = "https://api.github.com/user/repos?per_page=5"
    REPO_OWNER = ""   # auto-detected from /user response
    REPO_NAME  = "Databricks"
    REPO_URL   = ""   # built after /user call resolves the username
else:
    # Unauthenticated endpoints (60 req/hr, public repos only)
    GITHUB_USER_URL  = "https://api.github.com/users/firefighter7893-learner"
    GITHUB_REPOS_URL = "https://api.github.com/users/firefighter7893-learner/repos?per_page=5"
    REPO_OWNER = "firefighter7893-learner"
    REPO_NAME  = "Databricks"
    REPO_URL   = "https://api.github.com/repos/firefighter7893-learner/Databricks"


def get_github_headers(token):
    """Build GitHub API auth headers. Works with or without a token."""
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers

In [0]:
def test_endpoint(url, token, label=""):
    """Send an authenticated GET request to a GitHub API endpoint and report the result."""
    headers = get_github_headers(token)

    tag = f" [{label}]" if label else ""
    print(f"\n[ {datetime.now():%H:%M:%S} ]{tag} Testing: {url}")
    print(f"  Bearer Token: {'✅ Present (***{token[-4:] if len(token) > 4 else token})' if token else '❌ None (unauthenticated mode)'}")
    response = requests.get(url, headers=headers, timeout=30)

    print(f"  Status Code : {response.status_code}")
    print(f"  Content-Type: {response.headers.get('Content-Type', 'N/A')}")
    print(f"  Rate Limit  : {response.headers.get('X-RateLimit-Remaining', 'N/A')}/{response.headers.get('X-RateLimit-Limit', 'N/A')}")
    print(f"  Latency     : {response.elapsed.total_seconds():.3f}s")

    # Pretty-print a JSON body (truncated for readability)
    try:
        body = response.json()
        pretty = json.dumps(body, indent=2)
        if len(pretty) > 1000:
            pretty = pretty[:1000] + "\n  … (truncated)"
        print(f"  Body        :\n{pretty}")
    except Exception:
        print(f"  Body        : {response.text[:500]}")

    if response.status_code == 200:
        print(f"  ✅ {tag.strip()} PASSED")
    else:
        print(f"  ❌ {tag.strip()} FAILED (HTTP {response.status_code})")

    return response


def list_repo_contents(owner, repo, token=None, path=""):
    """List files and folders in a GitHub repository at the given path (root by default)."""
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
    headers = get_github_headers(token)

    print(f"\n📂 Repository: {owner}/{repo}" + (f"/{path}" if path else ""))
    print(f"   Endpoint : {url}")

    response = requests.get(url, headers=headers, timeout=30)

    if response.status_code != 200:
        print(f"   ❌ FAILED (HTTP {response.status_code}): {response.text[:200]}")
        return

    items = response.json()
    if not isinstance(items, list):
        items = [items]  # single file returned

    print(f"   Items found: {len(items)}")
    print()
    print(f"   {'Type':<10} {'Name':<40} {'Size (bytes)':>12}")
    print(f"   {'─'*10} {'─'*40} {'─'*12}")

    for item in items:
        item_type = "📁 dir" if item["type"] == "dir" else "📄 file"
        name = item["name"]
        size = item.get("size", "N/A")
        print(f"   {item_type:<10} {name:<40} {size:>12}")

    return response

In [0]:
# ──────────────────────────────────────────────
# Run GitHub API tests
# ──────────────────────────────────────────────
print("=" * 60)
print("  GitHub OAuth API Test")
print("=" * 60)

# Show bearer token status
print(f"\n🔑 Bearer Token Status:")
if GITHUB_TOKEN:
    print(f"   Token present: ***{GITHUB_TOKEN[-4:]}")
    print(f"   Mode: Authenticated (5,000 req/hr)")
else:
    print(f"   Token present: None")
    print(f"   Mode: Unauthenticated (60 req/hr)")

try:
    # 1. Authenticate — verify the token via /user endpoint
    user_resp = test_endpoint(GITHUB_USER_URL, GITHUB_TOKEN, label="Auth /user")

    # Auto-detect username from /user response (when authenticated)
    if GITHUB_TOKEN and user_resp.status_code == 200:
        REPO_OWNER = user_resp.json().get("login", "")
        REPO_URL = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}"
        print(f"\n   🔍 Auto-detected username: {REPO_OWNER}")
        print(f"   📦 Repo URL: {REPO_URL}")

    # 2. List your repositories (top 5)
    test_endpoint(GITHUB_REPOS_URL, GITHUB_TOKEN, label="Repos")

    # 3. Get a specific repository's details
    test_endpoint(REPO_URL, GITHUB_TOKEN, label="Repo Detail")

    # 4. List files and folders in the repository (root directory)
    print("\n" + "=" * 60)
    print("  📂 Repository Contents")
    print("=" * 60)
    list_repo_contents(REPO_OWNER, REPO_NAME, GITHUB_TOKEN)

    print("\n" + "=" * 60)
    print("  All tests complete.")
    print("=" * 60)

except Exception as e:
    print(f"\n❌ Unexpected error: {type(e).__name__}: {e}")